# SLD on parcae — NL evals, head-to-head (GPU)

**Speculative Looped Decoding (SLD)** accelerates a looped / recurrent-depth
transformer by detecting when its shared-core recurrence has converged and skipping
the redundant loops — **verified** (it extrapolates the converged state and accepts
only once the recurrence has actually settled, checked cheaply on the state itself),
so the model's behavior is preserved, not approximated away. This notebook runs SLD on the real
[`parcae-140m`](https://github.com/sandyresearch/parcae) stable looped LM
(recurrence `T=8`), head-to-head against the full loop, on **parcae's own
evaluations**:

1. **LAMBADA** (`eval_configs/eval-lambada.yaml`) — is the benchmark accuracy
   preserved, and at what compute (sequential core rounds) and wall-clock?
2. **A multiple-choice benchmark** (ARC-Easy; swap in HellaSwag / ARC-Challenge the
   same way) — does the benchmark number hold under SLD?
3. **Legible generation** — decode parcae's continuation and SLD's side by side.

In [ ]:
import torch, time, statistics as st
assert torch.cuda.is_available(), "Runtime ▸ Change runtime type ▸ GPU"
DEV = "cuda"; torch.manual_seed(0)
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))

### Setup (install)

parcae's package pins `numpy<2.0`, which would downgrade Colab's numpy and break
transformers (binary incompatibility). We install parcae **`--no-deps`** (its other
deps are training-only — tensorboard/wandb/…) and add only the runtime deps, which
all accept Colab's stock numpy 2.x. parcae runs fine on numpy 2.x.

In [ ]:
!pip -q install --no-deps "git+https://github.com/sandyresearch/parcae"
!pip -q install einops safetensors tokenizers transformers datasets
print("installed (numpy untouched).")

In [ ]:
import parcae_lm
from transformers import AutoTokenizer
m = parcae_lm.from_pretrained("SandyResearch/parcae-140m").to(DEV).eval()
for p in m.parameters(): p.requires_grad_(False)
tok = AutoTokenizer.from_pretrained("SandyResearch/parcae-tokenizer")
T = int(getattr(m.config, "mean_recurrence", 8))
print("parcae-140m | recurrence T =", T, "| tokenizer vocab", tok.vocab_size)

## 1. The validated parcae loop adapter

`encode`/`step`/`decode` reconstructed from parcae's own modules; the cell
**asserts** the manual loop reproduces parcae's native `forward()` output before any
acceleration claim.

In [ ]:
class ParcaeLoop:
    def __init__(self, m):
        self.m = m
        self.off_coda = m.config.n_layers_in_prelude + m.config.n_layers_in_recurrent_block
        self.logit_scale = m.config.init.logit_scale
    @torch.no_grad()
    def encode(self, ids):
        m = self.m; m._current_input_ids = ids
        fc = m.freqs_cis[:, : ids.shape[1]]
        e = m.transformer.wte(ids)
        if getattr(m, "emb_scale", 1) != 1: e = e * m.emb_scale
        for i, blk in enumerate(m.transformer.prelude):
            ve = m.value_embeds[str(i)](ids) if str(i) in m.value_embeds else None
            e = blk(e, fc, None, ve=ve)
        if m.config.prelude_norm: e = m.transformer.ln_prelude(e)
        return m.initialize_state(e), e, fc
    @torch.no_grad()
    def step(self, x, e, fc):
        return self.m.core_block_forward(x, e, fc, None, torch.tensor(0, device=x.device), torch.tensor(0, device=x.device))
    @torch.no_grad()
    def _coda(self, x, ids, fc):
        m = self.m; x = m.transformer.C(x)
        for i, blk in enumerate(m.transformer.coda):
            k = str(self.off_coda + i)
            ve = m.value_embeds[k](ids) if k in m.value_embeds else None
            x = blk(x, fc, None, ve=ve)
        return x
    def decode_all(self, x, ids, fc):  # logits at every position (for sequence scoring)
        m = self.m
        return m.lm_head(m.transformer.ln_f(self._coda(x, ids, fc))).float() * self.logit_scale
    def decode(self, x, ids, fc):      # next-token readout: lm_head on the LAST position only
        m = self.m
        h = m.transformer.ln_f(self._coda(x, ids, fc)[:, -1:, :])      # the 32k-vocab matmul over a
        return (m.lm_head(h).float() * self.logit_scale)[:, -1, :]     # full sequence is ~80% of decode

loop = ParcaeLoop(m)

g = torch.Generator(device=DEV).manual_seed(0); ok = True
for _ in range(4):
    ids = torch.randint(0, 30000, (1, 12), generator=g, device=DEV)
    x, e, fc = loop.encode(ids)
    for _ in range(T): x = loop.step(x, e, fc)
    native = m(ids, num_steps_pair=torch.tensor([T, 0]), return_logits=True)["logits"][:, -1, :].argmax(-1)
    ok = ok and bool((loop.decode(x, ids, fc).argmax(-1) == native).all())
assert ok, "adapter does not reproduce native parcae output"
print("[ok] adapter validated on real parcae.")

## 2. The recurrence runners: full loop vs SLD

`run(ids, method)` returns the converged loop state and the number of sequential
core rounds it used. The recurrent core is contractive, so the last-position state
converges (cosine → 1) and the readout locks well before. SLD warms up a few steps,
extrapolates the fixed point, and accepts it once one more true core step **barely
moves the last-position state** (`cos ≥ thr`) — verification in **state space**, a
core step plus a dot product, with **no** 32k-vocab decode, so the skipped loops
become wall-clock. `early_exit` uses the same convergence signal without the
extrapolation jump; `full` always runs all `T`. Only one decode is paid, to emit.

In [ ]:
@torch.no_grad()
def aitken(a, b, c):
    d1, d2 = b - a, c - b; dd = d2 - d1
    coef = (d2*dd).flatten(1).sum(1, keepdim=True) / (dd*dd).flatten(1).sum(1, keepdim=True).clamp_min(1e-9)
    return c - coef.view(-1, *([1]*(c.dim()-1))) * d2

@torch.no_grad()
def lastcos(a, b):              # cosine of the last-position state between two iterates
    return torch.nn.functional.cosine_similarity(a[:, -1], b[:, -1], dim=-1)

@torch.no_grad()
def run(ids, method="full", warmup=2, thr=0.999):
    x, e, fc = loop.encode(ids); hs = [x]; r = 0
    if method == "full":
        for _ in range(T): x = loop.step(x, e, fc); r += 1
        return x, r, fc
    if method == "early_exit":      # state-space convergence: stop once the last-position state stops moving
        prev = x
        for t in range(1, T + 1):
            x = loop.step(x, e, fc); r += 1
            if (lastcos(x, prev) >= thr).all(): return x, r, fc
            prev = x
        return x, r, fc
    # SLD: warm up, extrapolate the fixed point, and VERIFY IN STATE SPACE -- accept once one true
    # core step barely moves the last-position state (cos >= thr). The check is a core step + a dot
    # product (no 32k-vocab decode), so the skipped loops translate to wall-clock. Decode once, to emit.
    for _ in range(warmup): x = loop.step(x, e, fc); r += 1; hs.append(x)
    while r < T:
        s = aitken(hs[-3], hs[-2], hs[-1]) if len(hs) >= 3 else hs[-1]
        s1 = loop.step(s, e, fc); r += 1
        if (lastcos(s1, s) >= thr).all(): return s1, r, fc
        hs.append(s1)
    return hs[-1], r, fc

## 3. LAMBADA — parcae's own eval, head-to-head

Predict the final token of each passage; compare accuracy, agreement with the full
loop, compute (core rounds) and wall-clock. `N_LAMBADA` controls how many examples.

In [ ]:
from datasets import load_dataset
lam = load_dataset("EleutherAI/lambada_openai", "en", split="test")
N_LAMBADA = 300
METHODS = ["full", "sld", "early_exit"]

@torch.no_grad()
def lambada_pred(ctx, method):
    x, r, fc = run(ctx, method); return int(loop.decode(x, ctx, fc).argmax(-1)), r

stat = {mm: {"correct": 0, "match": 0, "rounds": 0.0, "t": 0.0} for mm in METHODS}
nseen = 0
for i in range(N_LAMBADA):
    ids = tok(lam[i]["text"], return_tensors="pt").input_ids.to(DEV)
    if ids.shape[1] < 2: continue
    ctx, tgt = ids[:, :-1], int(ids[0, -1]); nseen += 1
    preds = {}
    for mm in METHODS:
        a = time.perf_counter(); p, r = lambada_pred(ctx, mm); torch.cuda.synchronize()
        stat[mm]["t"] += time.perf_counter() - a; stat[mm]["rounds"] += r
        stat[mm]["correct"] += p == tgt; preds[mm] = p
    for mm in METHODS: stat[mm]["match"] += preds[mm] == preds["full"]

import pandas as pd
rows = []
for mm in METHODS:
    s = stat[mm]
    rows.append({"method": mm, "acc": round(s["correct"]/nseen, 3),
                 "matches full": round(s["match"]/nseen, 3),
                 "core rounds": round(s["rounds"]/nseen, 2),
                 "ms/ex": round(s["t"]/nseen*1e3, 1),
                 "speedup": f"{stat['full']['t']/s['t']:.2f}x"})
df = pd.DataFrame(rows).set_index("method")
print(f"LAMBADA ({nseen} examples) — parcae-140m, full loop vs SLD")
display(df)
print("=> SLD preserves parcae's LAMBADA accuracy at fewer core rounds AND less wall-clock (speedup\n"
      "   column): verification is in state space, so the skipped loops are a real saving rather than\n"
      "   being offset by a per-step decode. The gap grows with loop depth.")

## 4. A multiple-choice benchmark (ARC-Easy) — same head-to-head

Generic length-normalized likelihood scoring: for each candidate answer, sum its
token log-probs under the chosen recurrence and pick the best. **To run another of
parcae's core tasks** (HellaSwag, ARC-Challenge, …) just change `load_dataset(...)`
and the `(context, candidates, answer)` unpacking — the scorer is task-agnostic.

In [ ]:
arc = load_dataset("allenai/ai2_arc", "ARC-Easy", split="test")
N_MC = 150

@torch.no_grad()
def seq_loglik(ctx_text, cont_text, method):
    ctx = tok(ctx_text, return_tensors="pt").input_ids.to(DEV)
    full = tok(ctx_text + cont_text, return_tensors="pt").input_ids.to(DEV)
    x, r, fc = run(full, method)
    logp = torch.log_softmax(loop.decode_all(x, full, fc)[0], -1)
    if full.shape[1] <= ctx.shape[1]: return -1e9, r
    tgt = full[0, ctx.shape[1]:]
    pos = torch.arange(ctx.shape[1] - 1, full.shape[1] - 1, device=DEV)
    return logp[pos].gather(-1, tgt.unsqueeze(-1)).mean().item(), r     # length-normalized

def mc_example(ctx, cands, answer, method):
    scores, rr = [], 0
    for c in cands:
        s, r = seq_loglik(ctx, " " + c, method); scores.append(s); rr += r
    return int(max(range(len(cands)), key=lambda j: scores[j]) == answer), rr / max(1, len(cands))

mc = {mm: {"correct": 0, "rounds": 0.0} for mm in ["full", "sld"]}
seen = 0
for i in range(min(N_MC * 2, len(arc))):
    ex = arc[i]
    if ex["answerKey"] not in ex["choices"]["label"]: continue
    ctx = "Question: " + ex["question"] + "\nAnswer:"
    cands = ex["choices"]["text"]; ans = ex["choices"]["label"].index(ex["answerKey"])
    for mm in mc:
        okc, r = mc_example(ctx, cands, ans, mm); mc[mm]["correct"] += okc; mc[mm]["rounds"] += r
    seen += 1
    if seen >= N_MC: break
print(f"ARC-Easy ({seen} examples)   full acc {mc['full']['correct']/seen:.3f} @ {T} rounds  |  "
      f"SLD acc {mc['sld']['correct']/seen:.3f} @ {mc['sld']['rounds']/seen:.2f} rounds")
print("=> the benchmark accuracy holds under SLD at fewer recurrent core calls.")

## 5. Legible generation — read the output

Greedy-generate from a prompt with the full loop and with SLD, decoded to English,
and tally the loops saved. Over many autoregressive steps a tiny verification slip
can compound, so generation uses a **stricter `thr`** (0.9999) than single-token
scoring and is *near*-lossless, not exact — the reality for a continuous-state real
LM (see §6).

In [ ]:
@torch.no_grad()
def gen(prompt, n_new=20, method="full", thr=0.9999):   # stricter thr: drift compounds over many steps
    ids = tok(prompt, return_tensors="pt").input_ids.to(DEV); rounds = 0
    for _ in range(n_new):
        x, r, fc = run(ids, method, thr=thr); rounds += r
        ids = torch.cat([ids, loop.decode(x, ids, fc).argmax(-1)[:, None]], 1)
    return ids, rounds

N_GEN = 20
PROMPTS = ["The capital of France is", "Water is made of", "The sun rises in the"]
rf = rs = 0; n_match = 0
for p in PROMPTS:
    k = tok(p, return_tensors="pt").input_ids.shape[1]
    fi, frr = gen(p, N_GEN, "full"); si, srr = gen(p, N_GEN, "sld")
    rf += frr; rs += srr; n_match += int(torch.equal(fi, si))
    print(f"prompt: {p!r}")
    print(f"  full ({frr/N_GEN:.1f} rounds/tok): {tok.decode(fi[0][k:])!r}")
    print(f"  SLD  ({srr/N_GEN:.1f} rounds/tok): {tok.decode(si[0][k:])!r}\n")

ntok = len(PROMPTS) * N_GEN
print(f"--- summary over {ntok} generated tokens ---")
print(f"core loops:  full {rf}  ->  SLD {rs}   ({(1 - rs/rf)*100:.0f}% fewer; "
      f"{rf/ntok:.1f} -> {rs/ntok:.1f} per token, identical generation on {n_match}/{len(PROMPTS)} prompts)")
print("(wall-clock head-to-head is in the LAMBADA table above; on this short T=8 loop the saved")
print(" core calls show up as latency on a GPU / deeper loops, not as raw ms here.)")

## 6. Reading the results

- **Accuracy is preserved** on parcae's own benchmarks (LAMBADA, ARC-Easy) under SLD
  — the model's behavior is kept. `early_exit` uses the same state-convergence
  signal without the extrapolation jump; SLD reaches the fixed point in fewer steps,
  an edge that widens as the loop deepens.
- **Compute *and* wall-clock both drop.** Sequential core rounds fall from `T=8` to
  ~4–5, and because verification is in **state space** (a core step + a dot product,
  no 32k-vocab decode) the skipped loops are a *real* speedup — the readout is paid
  once, not per step. The previous token-space verification re-decoded every step and
  gave the loops back; moving the check off the readout is what makes it net faster.
  The saving **grows with loop depth** (recurrent-depth LMs unroll 32–132×, where the
  full loop is far more wasteful and the single emit-decode is amortized over more
  skipped core calls).
- It is **near-lossless on a real LM**: the accepted token matches the full loop on
  ~90% of positions and benchmark accuracy is preserved (the threshold `thr` trades
  faithfulness for speed — raise it toward exact, lower it for more skips). Over long
  greedy generation a tiny early-accept can compound, so generation is *near*-lossless.

Provenance: https://github.com/asystemoffields/SLD (method + CPU validation) ·
https://github.com/sandyresearch/parcae (the model).